In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

In [3]:
DATA_DIR = Path('/content/drive/MyDrive/Olist_Capstone/02_Cleaned_Data')

print("Folder exists:", DATA_DIR.exists())
print("\nCleaned CSV files:")

for file in sorted(DATA_DIR.glob("*.csv")):
    print(file.name)

Folder exists: True

Cleaned CSV files:
olist_customers_cleaned.csv
olist_geolocation_cleaned.csv
olist_order_items_cleaned.csv
olist_order_payments_cleaned.csv
olist_order_reviews_cleaned.csv
olist_orders_cleaned.csv
olist_products_cleaned.csv
olist_sellers_cleaned.csv
product_category_translation_cleaned.csv


###  Loading All Nine Cleaned Datasets

In this step, we load all nine cleaned CSV files into Pandas DataFrames. A DataFrame is a table-like structure that allows us to explore, analyse, and manipulate data using Python.

# Each dataset is stored in a separate variable so that we can perform individual and combined analysis in the following steps.

In [4]:
# Load all nine cleaned datasets

customers = pd.read_csv(DATA_DIR / "olist_customers_cleaned.csv")
geolocation = pd.read_csv(DATA_DIR / "olist_geolocation_cleaned.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_cleaned.csv")
order_payments = pd.read_csv(DATA_DIR / "olist_order_payments_cleaned.csv")
order_reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_cleaned.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_cleaned.csv")
products = pd.read_csv(DATA_DIR / "olist_products_cleaned.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_cleaned.csv")
category_translation = pd.read_csv(DATA_DIR / "product_category_translation_cleaned.csv")

print("All nine datasets loaded successfully!")

All nine datasets loaded successfully!


###  Checking Dataset Dimensions

In this step, we check the number of rows and columns in each dataset. This gives us an overview of the dataset sizes and helps confirm that the data has been loaded correctly.

In [5]:
datasets = {
    "Customers": customers,
    "Geolocation": geolocation,
    "Order Items": order_items,
    "Order Payments": order_payments,
    "Order Reviews": order_reviews,
    "Orders": orders,
    "Products": products,
    "Sellers": sellers,
    "Category Translation": category_translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows and {df.shape[1]} columns")

Customers: 99,441 rows and 5 columns
Geolocation: 738,332 rows and 5 columns
Order Items: 112,650 rows and 7 columns
Order Payments: 103,886 rows and 5 columns
Order Reviews: 99,224 rows and 7 columns
Orders: 99,441 rows and 8 columns
Products: 32,951 rows and 11 columns
Sellers: 3,095 rows and 4 columns
Category Translation: 71 rows and 2 columns


###  Dataset Summary

In this step, we create a summary of all nine datasets, including their names, number of rows, and number of columns. This provides a clear overview of the data used in the project and helps us understand the overall dataset structure.

In [6]:
dataset_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    }
    for name, df in datasets.items()
])

dataset_summary

,Dataset,Rows,Columns
0,Customers,99441,5
1,Geolocation,738332,5
2,Order Items,112650,7
3,Order Payments,103886,5
4,Order Reviews,99224,7
5,Orders,99441,8
6,Products,32951,11
7,Sellers,3095,4
8,Category Translation,71,2


In [7]:
for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(f"DATASET: {name}")
    print("=" * 50)

    print(df.dtypes)


DATASET: Customers
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

DATASET: Geolocation
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

DATASET: Order Items
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

DATASET: Order Payments
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

DATASET: Order Reviews
review_id                  object
order_id                   object
review_score    

###  Converting Date Columns into Datetime Format

In this step, we convert date-related columns into the proper datetime format using Pandas. This allows us to perform time-based analysis, calculate delivery durations, and study monthly and yearly sales trends more accurately.

The original CSV files will remain unchanged. These changes apply only to the DataFrames loaded into Python.

In [8]:
# Date columns in each dataset
date_columns = {
    "Orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "Order Items": [
        "shipping_limit_date"
    ],
    "Order Reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

# Convert date columns into datetime format

for dataset_name, columns in date_columns.items():
    df = datasets[dataset_name]

    for column in columns:
        df[column] = pd.to_datetime(df[column], errors="coerce")

print("Date columns converted successfully!")

Date columns converted successfully!


###  Verifying Date Conversions

After converting the date columns, we verify their data types to ensure that Pandas recognises them as datetime values. This confirms that the datasets are ready for time-based analysis.

In [9]:
for dataset_name, columns in date_columns.items():
    print(f"\n{dataset_name.upper()}")

    for column in columns:
        print(f"{column}: {datasets[dataset_name][column].dtype}")


ORDERS
order_purchase_timestamp: datetime64[ns]
order_approved_at: datetime64[ns]
order_delivered_carrier_date: datetime64[ns]
order_delivered_customer_date: datetime64[ns]
order_estimated_delivery_date: datetime64[ns]

ORDER ITEMS
shipping_limit_date: datetime64[ns]

ORDER REVIEWS
review_creation_date: datetime64[ns]
review_answer_timestamp: datetime64[ns]


### Checking Duplicate Records

In this step, we check all nine datasets for exact duplicate rows. Duplicate records can affect analysis results by causing incorrect counts and calculations. This verification helps ensure that the cleaned datasets do not contain repeated complete records.

This check does not remove or modify any records.

In [11]:
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()

    print(f"{name}: {duplicate_count:,} exact duplicate rows")

Customers: 0 exact duplicate rows
Geolocation: 0 exact duplicate rows
Order Items: 0 exact duplicate rows
Order Payments: 0 exact duplicate rows
Order Reviews: 0 exact duplicate rows
Orders: 0 exact duplicate rows
Products: 0 exact duplicate rows
Sellers: 0 exact duplicate rows
Category Translation: 0 exact duplicate rows


###  Order Status Distribution

In this step, we analyse the distribution of orders based on their current status. We calculate the number and percentage of orders in each category to understand the overall order fulfilment performance of the Olist platform.

In [12]:
# Count orders by status

order_status = orders["order_status"].value_counts().reset_index()

order_status.columns = ["Order Status", "Total Orders"]

# Calculate percentage

order_status["Percentage"] = (
    order_status["Total Orders"] / len(orders) * 100
).round(2)

order_status

,Order Status,Total Orders,Percentage
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


###  Monthly Order Trends

In this step, we analyse monthly order volumes using the purchase date. This helps identify seasonal patterns, busy periods, and changes in customer purchasing activity over time.

In [14]:
# Extract the month from the order purchase timestamp
orders["Order Month"] = (
    orders["order_purchase_timestamp"].dt.to_period("M")
)

# Count orders for each month
monthly_orders = (
    orders.groupby("Order Month")
    .size()
    .reset_index(name="Total Orders")
)

# Convert the month period into a string for visualisation
monthly_orders["Order Month"] = monthly_orders["Order Month"].astype(str)

monthly_orders

,Order Month,Total Orders
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


###  Visualising Monthly Order Trends

In this step, we create an interactive line chart to visualise monthly order volumes. This helps identify changes in customer purchasing activity and highlight the busiest months.

In [15]:
fig = px.line(
    monthly_orders,
    x="Order Month",
    y="Total Orders",
    title="Monthly Order Trends",
    markers=True,
    labels={
        "Order Month": "Month",
        "Total Orders": "Number of Orders"
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

###  Monthly Revenue Analysis

In this step, we calculate monthly product revenue using order purchase dates and order item prices. This helps identify high-performing sales periods and understand changes in revenue over time.

In [16]:
# Merge orders with order items
monthly_sales = order_items.merge(
    orders[["order_id", "order_purchase_timestamp"]],
    on="order_id",
    how="inner"
)

# Extract year and month
monthly_sales["Order Month"] = (
    monthly_sales["order_purchase_timestamp"].dt.to_period("M")
)

# Calculate monthly product revenue
monthly_revenue = (
    monthly_sales.groupby("Order Month")["price"]
    .sum()
    .reset_index(name="Total Revenue")
)

# Format month for display
monthly_revenue["Order Month"] = (
    monthly_revenue["Order Month"].astype(str)
)

monthly_revenue

,Order Month,Total Revenue
0,2016-09,267.36
1,2016-10,49507.66
2,2016-12,10.90
3,2017-01,120312.87
4,2017-02,247303.02
5,2017-03,374344.30
6,2017-04,359927.23
7,2017-05,506071.14
8,2017-06,433038.60
9,2017-07,498031.48


###  Visualising Monthly Revenue

In this step, we create an interactive line chart to visualise monthly product revenue. This helps identify revenue trends, highlight the highest-performing months, and compare sales performance over time.

In [17]:
fig = px.line(
    monthly_revenue,
    x="Order Month",
    y="Total Revenue",
    title="Monthly Revenue Trend",
    markers=True,
    labels={
        "Order Month": "Month",
        "Total Revenue": "Product Revenue (BRL)"
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    template="plotly_white",
    hovermode="x unified"
)

fig.update_traces(
    hovertemplate="Month: %{x}<br>Revenue: R$%{y:,.2f}<extra></extra>"
)

fig.show()

### Top 10 Product Categories by Revenue

In this step, we analyse product revenue across different categories. We identify the top 10 revenue-generating categories to understand which products contribute the most to the platform's overall sales.

In [19]:
# Merge order items with product details
category_sales = order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

# Merge with English category translations
category_sales = category_sales.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

# Replace missing category names
category_sales["product_category_name_english"] = (
    category_sales["product_category_name_english"]
    .fillna("Unknown")
)

# Calculate revenue by category
category_revenue = (
    category_sales.groupby("product_category_name_english")["price"]
    .sum()
    .reset_index(name="Total Revenue")
    .sort_values("Total Revenue", ascending=False)
)

# Select the top 10 categories
top_10_categories = category_revenue.head(10)

top_10_categories

,product_category_name_english,Total Revenue
44,health_beauty,1258681.34
71,watches_gifts,1205005.68
8,bed_bath_table,1036988.68
66,sports_leisure,988048.97
16,computers_accessories,911954.32
40,furniture_decor,729762.49
21,cool_stuff,635290.85
50,housewares,632248.66
6,auto,592720.11
43,garden_tools,485256.46


### Visualising Top 10 Product Categories by Revenue

In this step, we create an interactive horizontal bar chart to compare revenue across the top 10 product categories. This visualisation helps identify the strongest-performing categories and makes revenue comparisons easier.

In [20]:
fig = px.bar(
    top_10_categories.sort_values("Total Revenue"),
    x="Total Revenue",
    y="product_category_name_english",
    orientation="h",
    title="Top 10 Product Categories by Revenue",
    labels={
        "Total Revenue": "Product Revenue (BRL)",
        "product_category_name_english": "Product Category"
    },
    text="Total Revenue"
)

fig.update_traces(
    texttemplate="R$%{text:,.0f}",
    textposition="outside",
    hovertemplate=(
        "Category: %{y}<br>"
        "Revenue: R$%{x:,.2f}<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Product Revenue (BRL)",
    yaxis_title="Product Category"
)

fig.show()

###  Top 10 Sellers by Revenue

In this step, we analyse revenue generated by individual sellers. We identify the top 10 sellers based on product sales to understand their contribution to the platform's overall revenue.

In [24]:
# Calculate revenue generated by each seller
seller_revenue = (
    order_items.groupby("seller_id")["price"]
    .sum()
    .reset_index(name="Total Revenue")
    .sort_values("Total Revenue", ascending=False)
)

# Select the top 10 sellers
top_10_sellers = seller_revenue.head(10)

# Add seller location details
top_10_sellers = top_10_sellers.merge(
    sellers[["seller_id", "seller_city", "seller_state"]],
    on="seller_id",
    how="left"
)

top_10_sellers

,seller_id,Total Revenue,seller_city,seller_state
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,guariba,SP
1,53243585a1d6dc2643021fd1853d8905,222776.05,lauro de freitas,BA
2,4a3ca9315b744ce9f8e9374361493884,200472.92,ibitinga,SP
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,sumare,SP
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,itaquaquecetuba,SP
5,7e93a43ef30c4f03f38b393420bc753a,176431.87,barueri,SP
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,piracicaba,SP
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53,sao paulo,SP
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,sao paulo,SP
9,955fee9216a65b617aa5c0531780ce60,135171.70,sao paulo,SP


### Visualising Top 10 Sellers by Revenue

In this step, we create an interactive horizontal bar chart to compare revenue generated by the top 10 sellers. This helps identify the highest-performing sellers and understand their contribution to platform sales.

In [25]:
# Create unique seller labels
top_10_sellers["Seller Location"] = (
    top_10_sellers["seller_city"].str.title()
    + ", "
    + top_10_sellers["seller_state"]
    + " ("
    + top_10_sellers["seller_id"].str[:6]
    + ")"
)

# Sort sellers by revenue
seller_chart_data = top_10_sellers.sort_values(
    "Total Revenue",
    ascending=True
)

# Create horizontal bar chart
fig = px.bar(
    seller_chart_data,
    x="Total Revenue",
    y="Seller Location",
    orientation="h",
    title="Top 10 Sellers by Revenue",
    labels={
        "Total Revenue": "Product Revenue (BRL)",
        "Seller Location": "Seller Location"
    },
    text="Total Revenue"
)

fig.update_traces(
    texttemplate="R$%{text:,.0f}",
    textposition="outside",
    hovertemplate=(
        "Seller: %{y}<br>"
        "Revenue: R$%{x:,.2f}<extra></extra>"
    ),
    cliponaxis=False
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Product Revenue (BRL)",
    yaxis_title="Seller Location",
    margin=dict(r=130)
)

fig.show()

### Delivery Performance Analysis

In this step, we analyse delivery performance by comparing the actual delivery date with the estimated delivery date. This helps measure the platform's delivery reliability and identify potential delays.

In [26]:
# Select delivered orders with valid delivery dates
delivery_data = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna()) &
    (orders["order_estimated_delivery_date"].notna())
].copy()

# Classify delivery performance
delivery_data["Delivery Performance"] = (
    delivery_data["order_delivered_customer_date"]
    <= delivery_data["order_estimated_delivery_date"]
).map({
    True: "On Time / Early",
    False: "Late"
})

# Count delivery performance categories
delivery_performance = (
    delivery_data["Delivery Performance"]
    .value_counts()
    .reset_index()
)

delivery_performance.columns = [
    "Delivery Performance",
    "Total Orders"
]

# Calculate percentages
delivery_performance["Percentage"] = (
    delivery_performance["Total Orders"]
    / delivery_performance["Total Orders"].sum()
    * 100
).round(2)

delivery_performance

,Delivery Performance,Total Orders,Percentage
0,On Time / Early,88644,91.89
1,Late,7826,8.11


###  Visualising Delivery Performance

In this step, we create an interactive donut chart to compare on-time and late deliveries. This visualisation provides a clear overview of delivery reliability and highlights the percentage of delayed orders.

In [27]:
fig = px.pie(
    delivery_performance,
    names="Delivery Performance",
    values="Total Orders",
    title="Delivery Performance Distribution",
    hole=0.45
)

fig.update_traces(
    textposition="inside",
    textinfo="percent+label",
    hovertemplate=(
        "Performance: %{label}<br>"
        "Orders: %{value:,}<br>"
        "Percentage: %{percent}<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

###  Customer Review Score Analysis

In this step, we analyse customer review scores to understand customer satisfaction. We calculate the number of reviews for each score from 1 to 5 and examine the overall distribution of customer feedback.

In [28]:
# Count customer reviews by score
review_scores = (
    order_reviews["review_score"]
    .value_counts()
    .sort_index()
    .reset_index()
)

review_scores.columns = [
    "Review Score",
    "Total Reviews"
]

# Calculate review percentages
review_scores["Percentage"] = (
    review_scores["Total Reviews"]
    / review_scores["Total Reviews"].sum()
    * 100
).round(2)

review_scores

,Review Score,Total Reviews,Percentage
0,1,11424,11.51
1,2,3151,3.18
2,3,8179,8.24
3,4,19142,19.29
4,5,57328,57.78


###  Visualising Customer Review Scores

In this step, we create an interactive bar chart to visualise the distribution of customer review scores. This helps compare customer satisfaction levels and identify the most common review ratings.

In [29]:
fig = px.bar(
    review_scores,
    x="Review Score",
    y="Total Reviews",
    title="Customer Review Score Distribution",
    text="Total Reviews",
    labels={
        "Review Score": "Customer Review Score",
        "Total Reviews": "Number of Reviews"
    }
)

fig.update_traces(
    texttemplate="%{text:,}",
    textposition="outside",
    hovertemplate=(
        "Review Score: %{x}<br>"
        "Total Reviews: %{y:,}<extra></extra>"
    ),
    cliponaxis=False
)

fig.update_layout(
    template="plotly_white",
    xaxis=dict(
        tickmode="linear",
        dtick=1
    ),
    yaxis_title="Number of Reviews"
)

fig.show()

### Payment Type Distribution

In this step, we analyse the distribution of payment methods used by customers. This helps identify the most preferred payment options and understand customer payment behaviour.

In [31]:
# Count payment records by payment type
payment_types = (
    order_payments["payment_type"]
    .value_counts()
    .reset_index()
)

payment_types.columns = [
    "Payment Type",
    "Total Payments"
]

# Calculate percentages
payment_types["Percentage"] = (
    payment_types["Total Payments"]
    / payment_types["Total Payments"].sum()
    * 100
).round(2)

payment_types

,Payment Type,Total Payments,Percentage
0,credit_card,76795,73.92
1,boleto,19784,19.04
2,voucher,5775,5.56
3,debit_card,1529,1.47
4,not_defined,3,0.00


### Visualising Payment Type Distribution

In this step, we create an interactive donut chart to visualise the distribution of payment methods. This helps identify customer payment preferences and compare the popularity of different payment options.

In [32]:
fig = px.pie(
    payment_types,
    names="Payment Type",
    values="Total Payments",
    title="Payment Type Distribution",
    hole=0.45
)

fig.update_traces(
    textposition="inside",
    textinfo="percent+label",
    hovertemplate=(
        "Payment Method: %{label}<br>"
        "Total Payments: %{value:,}<br>"
        "Percentage: %{percent}<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white"
)

fig.show()